## Imports

In [13]:
import pandas as pd
import json
import time
import random
import requests
from dotenv import load_dotenv
import os
from tqdm import tqdm
load_dotenv()

True

## Configs

In [14]:
#==Config==#
OPEN_ROUTER_API_KEY = os.getenv("OPEN_ROUTER_API_KEY")

## Load Data

In [15]:
df = pd.read_csv("D:\Work Docs\BE Project\TwinSpherev2\PreprocessingScripts\grouped_concatenated_tweets.csv")

In [16]:
df.head()

,name,concatenated_tweets
0,10Ronaldinho,b'Baita recep\xc3\xa7\xc3\xa3o dessa galera an...
1,10neto,b'@andreolifelipe A\xc3\xad brigam com a gente...
2,143redangel,b'Julie Yap-Daza = huli (To get caught). \n\ne...
3,1LoganHenderson,"b""Check this out! @LOWDNoizez' debut single of..."
4,1victorvaldes,b'\xd8\xb4\xd8\xb1\xd9\x83\xd8\xa9 \xd8\xa7\xd...


## Helper Functions

In [17]:
# Prompt Template
def build_prompt(tweet_block):
    return f"""
Analyze the following Twitter content from a user. Based on their language, common themes, and overall behavior expressed in these tweets, infer the following personality traits. Output the results as a detailed persona paragraph.

1.  **Interests/Topics:** List 3-5 keywords or phrases describing their main interests and the topics they discuss.
2.  **Communication Style & Tone:** Describe how they express themselves.
3.  **General Sentiment:** Describe their overall emotional leaning.
4.  **Engagement Patterns:** Describe how they interact on the platform.

Ensure your output is a paragraph detailing the persona, using the points mentioned above. There should be nothing apart from this paragraph in your output.

The following Content section contains the latest 100 tweets of the user concatenated together as a string and seperated by the seperator "<ENDOFTWEET>"

Content: {tweet_block}
""".strip()

In [18]:
# Function to call OpenRouter with DeepSeek
def query_openrouter(prompt, model="deepseek/deepseek-chat-v3-0324:free"):
    response = requests.post(
        url="https://openrouter.ai/api/v1/chat/completions",
        headers={
            "Authorization": f"Bearer {OPEN_ROUTER_API_KEY}",
            "Content-Type": "application/json"
        },
        data=json.dumps({
            "model": "deepseek/deepseek-chat-v3-0324:free",
            "messages": [{
                "role": "user",
                "content": prompt
            }],
            
        })
    )

    if response.status_code != 200:
        raise Exception(f"API error {response.status_code}: {response.text}")

    result = response.json()
    return result["choices"][0]["message"]["content"]

## Extract persona

In [19]:
output_data = []

sampled_df = df.sample(10)

# Iterate through rows and generate personas
for _, row in tqdm(sampled_df.iterrows(), total=len(sampled_df), desc="Extracting Personas"):
    name = row["name"]
    tweets = row["concatenated_tweets"]

    try:
        prompt = build_prompt(tweets)
        response = query_openrouter(prompt)
        output_data.append({"name": name, "persona": response})
        time.sleep(1.5)  # Avoid rate-limiting
    except Exception as e:
        print(f"[!] Error processing {name}: {e}")
        continue

Extracting Personas:  50%|███████████████████████████████                               | 5/10 [00:00<00:00, 21.11it/s]

[!] Error processing adamlambert: API error 401: {"error":{"message":"Missing Authentication header","code":401}}
[!] Error processing xuxameneghel: API error 401: {"error":{"message":"Missing Authentication header","code":401}}
[!] Error processing daddy_yankee: API error 401: {"error":{"message":"Missing Authentication header","code":401}}
[!] Error processing VINNYGUADAGNINO: API error 401: {"error":{"message":"Missing Authentication header","code":401}}
[!] Error processing brookeburke: API error 401: {"error":{"message":"Missing Authentication header","code":401}}


Extracting Personas: 100%|█████████████████████████████████████████████████████████████| 10/10 [00:00<00:00, 21.97it/s]

[!] Error processing jeremypiven: API error 401: {"error":{"message":"Missing Authentication header","code":401}}
[!] Error processing jamesmaslow: API error 401: {"error":{"message":"Missing Authentication header","code":401}}
[!] Error processing coldplay: HTTPSConnectionPool(host='openrouter.ai', port=443): Max retries exceeded with url: /api/v1/chat/completions (Caused by SSLError(SSLError(1, '[SSL: SSLV3_ALERT_BAD_RECORD_MAC] sslv3 alert bad record mac (_ssl.c:2580)')))
[!] Error processing TomCavalcante1: API error 401: {"error":{"message":"Missing Authentication header","code":401}}
[!] Error processing mnzadornov: HTTPSConnectionPool(host='openrouter.ai', port=443): Max retries exceeded with url: /api/v1/chat/completions (Caused by SSLError(SSLError(1, '[SSL: SSLV3_ALERT_BAD_RECORD_MAC] sslv3 alert bad record mac (_ssl.c:2580)')))


In [20]:
pd.DataFrame(output_data).to_csv("D:\Projects\Personal\Hackathon\TwinSpherev2\Data\Persona.csv")

OSError: Cannot save file into a non-existent directory: 'D:\Projects\Personal\Hackathon\TwinSpherev2\Data'